# 01 — Inventory data-contract audit

The backend loader is the canonical preprocessing implementation. Run `python src/backend/main.py` from the repository root to regenerate validated artifacts. This notebook is read-only: it compares the raw extract, processed dataset, metadata contract, and artifact hashes without repairing or persisting rows.

Core grain: one row per `sku_id` per calendar date. `inventory_level` is end-of-day on-hand inventory.

In [ ]:
from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "src" / "backend").exists():
    ROOT = ROOT.parent
if not (ROOT / "src" / "backend").exists():
    raise FileNotFoundError("Open this notebook from the repository root or notebooks directory.")

PATHS = {
    "raw": ROOT / "data" / "raw" / "inventory.csv",
    "processed": ROOT / "data" / "processed" / "inventory_processed.csv",
    "contract": ROOT / "data" / "metadata" / "inventory_data_contract.json",
    "manifest": ROOT / "data" / "metadata" / "artifact_manifest.json",
}
missing = [str(path.relative_to(ROOT)) for path in PATHS.values() if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Missing canonical data artifacts. Run `python src/backend/main.py`: " + ", ".join(missing)
    )

raw = pd.read_csv(PATHS["raw"], parse_dates=["date"])
processed = pd.read_csv(PATHS["processed"], parse_dates=["date"])
contract = json.loads(PATHS["contract"].read_text(encoding="utf-8"))
manifest = json.loads(PATHS["manifest"].read_text(encoding="utf-8"))

{
    "raw_shape": raw.shape,
    "processed_shape": processed.shape,
    "contract_grain": contract["grain"],
    "contract_target": contract["target"],
}

## Schema and primary-key checks

Validation is fail-fast: missing values, duplicate SKU-date keys, or silently changed columns are contract violations rather than rows to drop.

In [ ]:
schema_audit = pd.DataFrame(
    [
        {
            "column": column,
            "expected_dtype": expected_dtype,
            "actual_dtype": str(processed[column].dtype) if column in processed.columns else None,
            "present": column in processed.columns,
            "dtype_matches": column in processed.columns and str(processed[column].dtype) == expected_dtype,
        }
        for column, expected_dtype in contract["schema"].items()
    ]
)
primary_key = contract["primary_key"]
key_audit = pd.DataFrame(
    [
        {
            "dataset": label,
            "rows": len(frame),
            "skus": frame["sku_id"].nunique(),
            "missing_key_rows": int(frame[primary_key].isna().any(axis=1).sum()),
            "duplicate_key_rows": int(frame.duplicated(primary_key, keep=False).sum()),
            "missing_value_cells": int(frame.isna().sum().sum()),
        }
        for label, frame in [("raw", raw), ("processed", processed)]
    ]
)
raw_normalized = raw.sort_values(primary_key).reset_index(drop=True)
processed_normalized = processed.sort_values(primary_key).reset_index(drop=True)
content_audit = pd.DataFrame(
    [
        {
            "check": "raw and processed columns",
            "passed": raw_normalized.columns.tolist() == processed_normalized.columns.tolist(),
        },
        {
            "check": "raw and processed validated values",
            "passed": raw_normalized.equals(processed_normalized),
        },
    ]
)
display(schema_audit)
display(key_audit)
display(content_audit)

In [ ]:
ordered = processed.sort_values(["sku_id", "date"]).reset_index(drop=True)
previous_date = ordered.groupby("sku_id", observed=True)["date"].shift(1)
day_gap = (ordered["date"] - previous_date).dt.days
calendar_by_sku = (
    ordered.groupby("sku_id", observed=True)
    .agg(start=("date", "min"), end=("date", "max"), rows=("date", "size"), unique_dates=("date", "nunique"))
    .reset_index()
)
calendar_by_sku["expected_days"] = (calendar_by_sku["end"] - calendar_by_sku["start"]).dt.days + 1
calendar_by_sku["continuous"] = (
    calendar_by_sku["rows"].eq(calendar_by_sku["expected_days"])
    & calendar_by_sku["rows"].eq(calendar_by_sku["unique_dates"])
)
calendar_audit = pd.DataFrame(
    [
        {"check": "all dates are normalized midnight", "passed": bool(ordered["date"].dt.normalize().eq(ordered["date"]).all())},
        {"check": "all within-SKU gaps equal one day", "passed": bool(day_gap.dropna().eq(1).all())},
        {"check": "every SKU has a continuous calendar", "passed": bool(calendar_by_sku["continuous"].all())},
        {"check": "row count matches contract", "passed": len(processed) == contract["row_count"]},
        {"check": "SKU count matches contract", "passed": processed["sku_id"].nunique() == contract["sku_count"]},
        {"check": "start date matches contract", "passed": processed["date"].min().date().isoformat() == contract["date_range"]["start"]},
        {"check": "end date matches contract", "passed": processed["date"].max().date().isoformat() == contract["date_range"]["end"]},
    ]
)
display(calendar_audit)
display(calendar_by_sku.head())

In [ ]:
previous_inventory = ordered.groupby("sku_id", observed=True)["inventory_level"].shift(1)
observable_transition = previous_inventory.notna()
expected_end_inventory = (
    previous_inventory[observable_transition]
    + ordered.loc[observable_transition, "order_received"]
    - ordered.loc[observable_transition, "sales_quantity"]
)
conservation_violation = ~ordered.loc[observable_transition, "inventory_level"].eq(expected_end_inventory)
nonnegative_columns = [
    "demand",
    "sales_quantity",
    "inventory_level",
    "order_received",
    "lead_time",
    "safety_stock",
    "reorder_point",
    "order_quantity",
    "unit_cost",
    "unit_price",
]
operational_audit = pd.DataFrame(
    [
        {"check": "end-of-day inventory conservation", "violations": int(conservation_violation.sum())},
        {"check": "nonnegative operational values", "violations": int((ordered[nonnegative_columns] < 0).sum().sum())},
        {"check": "sales do not exceed latent demand", "violations": int((ordered["sales_quantity"] > ordered["demand"]).sum())},
        {"check": "lead time is positive integer days", "violations": int((ordered["lead_time"] <= 0).sum() + (~np.isclose(ordered["lead_time"], np.round(ordered["lead_time"]))).sum())},
    ]
)
operational_audit

In [ ]:
def sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

manifest_artifacts = manifest.get("artifacts", {})
hash_audit = pd.DataFrame(
    [
        {
            "artifact": path.relative_to(ROOT).as_posix(),
            "actual_sha256": sha256(path),
            "manifest_sha256": manifest_artifacts.get(path.relative_to(ROOT).as_posix(), {}).get("sha256"),
        }
        for path in [PATHS["raw"], PATHS["processed"], PATHS["contract"]]
    ]
)
hash_audit["matches_manifest"] = hash_audit["actual_sha256"].eq(hash_audit["manifest_sha256"])
hash_audit

## Latent-demand semantics

`demand` is unconstrained unit demand, while `sales_quantity` is fulfilled demand capped by available inventory. Their difference is the modeled lost-sale quantity. This target is observable here because the dataset is synthetic. In a real ERP extract, sales during a stockout are censored and cannot be relabeled as unconstrained demand without an explicit lost-demand estimation method.

In [ ]:
demand_gap = processed["demand"] - processed["sales_quantity"]
latent_demand_audit = pd.DataFrame(
    [
        {"measure": "total latent demand units", "value": float(processed["demand"].sum())},
        {"measure": "total fulfilled units", "value": float(processed["sales_quantity"].sum())},
        {"measure": "modeled lost-demand units", "value": float(demand_gap.sum())},
        {"measure": "SKU-days with constrained sales", "value": int(demand_gap.gt(0).sum())},
        {"measure": "rows with impossible negative demand gap", "value": int(demand_gap.lt(0).sum())},
    ]
)
latent_demand_audit